<a href="https://colab.research.google.com/github/yashwini321/yashwini/blob/main/spam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models
df = pd.read_csv("/content/Spam_Detection.csv")
df = df.dropna()
df["Label"] = df["Label"].map({
    "ham": 0,
    "spam": 1
})
texts = df["Mail"].astype(str).values
labels = df["Label"].values
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)
MAX_WORDS = 10000
MAX_LENGTH = 100
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)
X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

model = models.Sequential([
    layers.Embedding(
        input_dim=MAX_WORDS,
        output_dim=128,
        input_length=MAX_LENGTH
    ),
    layers.Bidirectional(
        layers.LSTM(64)
    ),
    layers.Dropout(0.5),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
model.summary()

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping]
)

loss, accuracy = model.evaluate(
    X_test_pad,
    y_test
)
print("Test Accuracy:", accuracy)

predictions = model.predict(X_test_pad)
y_pred = (predictions >= 0.5).astype(int).flatten()

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Spam", "Spam"]
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

model.save("spam_lstm.keras")

import pickle
with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

print("Model and tokenizer saved.")
import streamlit as st
import tensorflow as tf
import numpy as np
import pickle
from tensorflow.keras.preprocessing.sequence import pad_sequences
model = tf.keras.models.load_model(
    "spam_lstm.keras"
)
with open("tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

MAX_LENGTH = 100
st.set_page_config(
    page_title="Spam Detector",
    page_icon="📩"
)

st.title("📩 Real-Time Spam Detection")

st.write(
    "Enter an SMS, email, or message to determine "
    "whether it is spam."
)

message = st.text_area(
    "Enter your message:",
    height=150,
    placeholder="Type your message here..."
)
if st.button("🔍 Check Message"):

    if message.strip() == "":
        st.warning("Please enter a message.")

    else:


        sequence = tokenizer.texts_to_sequences(
            [message]
        )

        padded = pad_sequences(
            sequence,
            maxlen=MAX_LENGTH,
            padding="post",
            truncating="post"
        )

        prediction = model.predict(
            padded,
            verbose=0
        )[0][0]

        spam_probability = prediction * 100
        ham_probability = (1 - prediction) * 100


        if prediction >= 0.5:

            st.error("🚨 SPAM MESSAGE")

            st.metric(
                "Spam Probability",
                f"{spam_probability:.2f}%"
            )

        else:

            st.success("✅ NOT SPAM")

            st.metric(
                "Not Spam Probability",
                f"{ham_probability:.2f}%"
            )


        st.subheader("Prediction Probability")

        st.write(
            f"Spam: {spam_probability:.2f}%"
        )

        st.progress(
            float(prediction)
        )

        st.write(
            f"Not Spam: {ham_probability:.2f}%"
        )

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 21s 152ms/step - accuracy: 0.9276 - loss: 0.2097 - val_accuracy: 0.9832 - val_loss: 0.0565
Epoch 2/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 17s 153ms/step - accuracy: 0.9902 - loss: 0.0445 - val_accuracy: 0.9832 - val_loss: 0.0570
Epoch 3/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 21s 155ms/step - accuracy: 0.9964 - loss: 0.0181 - val_accuracy: 0.9821 - val_loss: 0.0702
Epoch 4/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 17s 152ms/step - accuracy: 0.9975 - loss: 0.0127 - val_accuracy: 0.9843 - val_loss: 0.0747
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.9839 - loss: 0.0688
Test Accuracy: 0.9838564991950989
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step
              precision    recall  f1-score   support

    Not Spam       0.99      0.99      0.99       966
        Spam       0.96      0.91      0.94       149

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115

C

ModuleNotFoundError: No module named 'streamlit'